In [1]:
import pandas as pd

In [2]:
# Load the CRSP characteristics dataset from the paper

datashare_df = pd.read_csv("/Users/shadowform/Documents/#Unito/CodesAndProjects/EquityCharacteristics-master/datashare/datashare.csv")
print(f"Datashare dataset loaded successfully.\n{'-' * 40}")
print(f"Observations: {datashare_df.shape[0]:,}\nVariables:    {datashare_df.shape[1]:,}")

Datashare dataset loaded successfully.
----------------------------------------
Observations: 4,117,300
Variables:    97


In [3]:
# Load the CRSP return dataset from WRDS

return_df = pd.read_csv("data_csv/CRSP_returns.csv")
print(f"WRDS dataset loaded successfully.\n{'-' * 40}")
print(f"Observations: {return_df.shape[0]:,}\nColumns:    {return_df.shape[1]:,}")


WRDS dataset loaded successfully.
----------------------------------------
Observations: 4,987,869
Columns:    10


In [4]:
# How many unique keys match

crsp_id_column = "PERMNO"
crsp_date_column = "MthCalDt"
ds_id_column = "permno"
ds_date_column = "DATE"

# Build comparable (permno, date) keys in both datasets with the same column names
crsp_keys = return_df[[crsp_id_column, crsp_date_column]].copy()
crsp_keys[crsp_date_column] = pd.to_datetime(crsp_keys[crsp_date_column], errors="coerce")
crsp_keys = crsp_keys.rename(columns={crsp_id_column: "permno", crsp_date_column: "date"})
crsp_keys = crsp_keys.dropna()


ds_keys = datashare_df[[ds_id_column, ds_date_column]].copy()
ds_keys[ds_date_column] = pd.to_datetime(ds_keys[ds_date_column].astype(str), format="%Y%m%d", errors="coerce")
ds_keys = ds_keys.rename(columns={ds_id_column: "permno", ds_date_column: "date"})
ds_keys = ds_keys.dropna()

# Count occurrences of each (permno, date) pair
crsp_pair_counts = crsp_keys.groupby(["permno", "date"]).size()
datashare_pair_counts = ds_keys.groupby(["permno", "date"]).size()

# Number of unique matching (permno, date) pairs
common_unique_pairs = crsp_pair_counts.index.intersection(datashare_pair_counts.index)

print(f"CRSP unique pairs: {len(crsp_pair_counts):,}")
print(f"Datashare unique pairs: {len(datashare_pair_counts):,}")
print("-" * 40)
print(f"Final dataset unique pairs: {len(common_unique_pairs):,}")
print(f"Overlap ratio (CRSP): {len(common_unique_pairs) / len(crsp_pair_counts):.2%}")
print(f"Overlap ratio (DS): {len(common_unique_pairs) / len(datashare_pair_counts):.2%}")
print("-" * 40)
print(f"So we'll have {len(common_unique_pairs):,} rows after merge and can be used for later model building")

CRSP unique pairs: 4,929,752
Datashare unique pairs: 4,117,300
----------------------------------------
Final dataset unique pairs: 4,117,032
Overlap ratio (CRSP): 83.51%
Overlap ratio (DS): 99.99%
----------------------------------------
So we'll have 4,117,032 rows after merge and can be used for later model building


In [ ]:
# Copies so you do not overwrite the raw dataframes
crsp = return_df.copy()
ds = datashare_df.copy()

# 1. Show variable names clearly
crsp_cols = pd.Index(crsp.columns)
ds_cols = pd.Index(ds.columns)

print("CRSP columns")
display(pd.DataFrame({"crsp_columns": crsp_cols}))

print("Datashare columns")
display(pd.DataFrame({"datashare_columns": ds_cols}))

CRSP columns


,crsp_columns
0,PERMNO
1,HdrCUSIP
2,Ticker
3,PERMCO
4,MthCalDt
5,MthPrc
6,MthRet
7,MthRetx
8,MthVol
9,ShrOut


Datashare columns


,datashare_columns
0,permno
1,DATE
2,mvel1
3,beta
4,betasq
...,...
92,retvol
93,std_dolvol
94,std_turn
95,zerotrade
